# Fine-tune Qwen3-1.7B on the Minecraft bot's decisions (free Colab GPU)

What this does: teaches the small `qwen3:1.7b` to make the bot's decisions the way its best recorded decisions did, using the **short prompt** (`llm.trained`), so it decides several times faster on a 6 GB card.

You need `train-short.jsonl` and `test-short.jsonl` from your PC (`npm run training-set -- --short`, in `data/training/`). See TRAINING.md for how to record good ones.

**Steps:** Runtime → Change runtime type → **T4 GPU**. Then run the cells from top to bottom (Shift+Enter). About 30–90 minutes for a few thousand examples.

If a cell errors after an Unsloth update, open Unsloth's current Qwen3 notebook (unsloth.ai → Notebooks) and copy that cell's version; the steps stay the same.

In [ ]:
# 1. Install Unsloth (a few minutes)
!pip install -q unsloth
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none: switch the runtime to T4 GPU')

In [ ]:
# 2. Upload train-short.jsonl and test-short.jsonl (from data/training on your PC)
from google.colab import files
uploaded = files.upload()
print(list(uploaded))

In [ ]:
# 3. Load the model in 4-bit and add LoRA adapters (only these small extra weights are trained)
from unsloth import FastLanguageModel
MAX_LEN = 4096  # the short prompt + state is ~1,500 tokens; room to spare
model, tokenizer = FastLanguageModel.from_pretrained(model_name='unsloth/Qwen3-1.7B', max_seq_length=MAX_LEN, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=32, lora_alpha=32, lora_dropout=0, bias='none',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth', random_state=3407)

In [ ]:
# 4. The examples, written the way Ollama will show them to the model (no thinking, like llm.think: false)
from datasets import load_dataset
train = load_dataset('json', data_files='train-short.jsonl', split='train')
test = load_dataset('json', data_files='test-short.jsonl', split='train')
def to_text(batch):
    return {'text': [tokenizer.apply_chat_template(m, tokenize=False, enable_thinking=False) for m in batch['messages']]}
train = train.map(to_text, batched=True)
test_text = test.map(to_text, batched=True)
lengths = [len(tokenizer(t).input_ids) for t in train['text'][:200]]
print(len(train), 'examples; tokens each (first 200): avg', sum(lengths)//len(lengths), 'max', max(lengths))

In [ ]:
# 5. Train. Only the answers are learned (not the prompts). 1 epoch first; 2 if the test below is weak.
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=train, eval_dataset=test_text,
    args=SFTConfig(dataset_text_field='text', max_seq_length=MAX_LEN, per_device_train_batch_size=2, gradient_accumulation_steps=4,
                   warmup_steps=10, num_train_epochs=1, learning_rate=2e-4, logging_steps=10, optim='adamw_8bit',
                   weight_decay=0.01, lr_scheduler_type='linear', seed=3407, output_dir='outputs', report_to='none'))
trainer = train_on_responses_only(trainer, instruction_part='<|im_start|>user\n', response_part='<|im_start|>assistant\n')
stats = trainer.train()
print(stats)

In [ ]:
# 6. Test it on decisions it never saw: valid JSON, same action, same action and target
import json, re
FastLanguageModel.for_inference(model)
MAIN = ['blockType', 'itemName', 'playerName', 'ore', 'entityId', 'topic', 'direction']
def main_arg(a):
    for k in MAIN:
        if isinstance(a, dict) and k in a: return f'{k}={str(a[k]).lower()}'
    return ''
valid = same = same_args = 0
N = min(40, len(test))
for ex in test.select(range(N)):
    msgs = [m for m in ex['messages'] if m['role'] != 'assistant']
    want = json.loads([m for m in ex['messages'] if m['role'] == 'assistant'][0]['content'])
    ids = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, enable_thinking=False, return_tensors='pt').to('cuda')
    out = model.generate(input_ids=ids, max_new_tokens=200, temperature=0.2, do_sample=True)
    text = tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
    text = re.sub(r'^[\s\S]*</think>', '', text).strip()
    try:
        got = json.loads(text[text.index('{'):text.rindex('}') + 1])
    except Exception:
        continue
    if got.get('action', {}).get('name'): valid += 1
    if got.get('action', {}).get('name') == want['action']['name']:
        same += 1
        if main_arg(got['action'].get('args')) == main_arg(want['action'].get('args')): same_args += 1
print(f'valid {valid}/{N} · same action {same}/{N} · same action and target {same_args}/{N}')
print('Aim for valid ~100% and same action 60-80%. Lower: record more (and better) examples, or train a 2nd epoch.')

In [ ]:
# 7. Export to a GGUF file Ollama can run (q4_k_m: small and fast), then download it (~1.1 GB)
model.save_pretrained_gguf('minecraft-bot', tokenizer, quantization_method='q4_k_m')
import glob
gguf = sorted(glob.glob('minecraft-bot*/*.gguf') + glob.glob('*.gguf'))
print(gguf)
files.download(gguf[-1])

## On your PC

1. Put the downloaded `.gguf` in a folder, and next to it a file named `Modelfile` containing one line: `FROM ./<the file name>.gguf`
2. In that folder: `ollama create minecraft-bot -f Modelfile`
3. Compare it with the old model on the held-back decisions: `npm run eval-model -- minecraft-bot` and `npm run eval-model -- qwen3:4b-instruct`
4. Use it: `npm run set -- llm.model minecraft-bot` and `npm run set -- llm.trained true` (and `npm run set -- llm.fastModel ""`), then restart the bot and watch the `Thought for … s` lines.
5. If it plays worse than the old model: `npm run set -- llm.model qwen3:4b-instruct` and `npm run set -- llm.trained false` put everything back.